AIM

To create a DataFrame using PySpark and perform data cleaning and feature processing on the dataset.

THEORY

PySpark is a Python API for Apache Spark that is used for processing and analyzing large datasets efficiently. A PySpark DataFrame is a distributed, table-like data structure consisting of rows and columns.

Data cleaning is the process of improving data quality by identifying and handling problems such as missing values, duplicate records, and inconsistent data. In PySpark, functions such as fillna() and dropDuplicates() can be used for data cleaning.

Feature processing involves transforming existing data or creating new features that are useful for analysis and machine learning. In this practical, missing values were replaced using suitable values, duplicate records were removed, and new features such as salary_in_thousands and age_group were created.

Finally, quality checks were performed to verify that the processed data contained no missing values. The final cleaned and processed DataFrame was then exported as a CSV file.

In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Data Cleaning and Feature Processing") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


In [ ]:
from pyspark.sql import Row

data = [
    (1, "Amit", 20, "Pune", 45000),
    (2, "Pooja", 21, "Mumbai", 50000),
    (3, "Rahul", None, "Pune", 55000),
    (4, "Neha", 22, None, 48000),
    (5, "Vikas", 20, "Nashik", None),
    (6, "Sneha", 23, "Pune", 60000),
    (6, "Sneha", 23, "Pune", 60000)
]

columns = ["id", "name", "age", "city", "salary"]

df = spark.createDataFrame(data, columns)

df.show()

+---+-----+----+------+------+
| id| name| age|  city|salary|
+---+-----+----+------+------+
|  1| Amit|  20|  Pune| 45000|
|  2|Pooja|  21|Mumbai| 50000|
|  3|Rahul|NULL|  Pune| 55000|
|  4| Neha|  22|  NULL| 48000|
|  5|Vikas|  20|Nashik|  NULL|
|  6|Sneha|  23|  Pune| 60000|
|  6|Sneha|  23|  Pune| 60000|
+---+-----+----+------+------+



In [ ]:
from pyspark.sql.functions import col, sum

# Display schema
df.printSchema()

# Count missing values in each column
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: long (nullable = true)

+---+----+---+----+------+
| id|name|age|city|salary|
+---+----+---+----+------+
|  0|   0|  1|   1|     1|
+---+----+---+----+------+



In [ ]:
from pyspark.sql.functions import avg

# Calculate averages
avg_age = df.select(avg("age")).first()[0]
avg_salary = df.select(avg("salary")).first()[0]

# Fill missing values
df = df.fillna({
    "age": int(avg_age),
    "city": "Unknown",
    "salary": int(avg_salary)
})

print("Missing values handled successfully.")
df.show()

Missing values handled successfully.
+---+-----+---+-------+------+
| id| name|age|   city|salary|
+---+-----+---+-------+------+
|  1| Amit| 20|   Pune| 45000|
|  2|Pooja| 21| Mumbai| 50000|
|  3|Rahul| 21|   Pune| 55000|
|  4| Neha| 22|Unknown| 48000|
|  5|Vikas| 20| Nashik| 53000|
|  6|Sneha| 23|   Pune| 60000|
|  6|Sneha| 23|   Pune| 60000|
+---+-----+---+-------+------+



In [ ]:
# Remove duplicate records
df = df.dropDuplicates()

print("Duplicate records removed successfully.")
df.show()

Duplicate records removed successfully.
+---+-----+---+-------+------+
| id| name|age|   city|salary|
+---+-----+---+-------+------+
|  3|Rahul| 21|   Pune| 55000|
|  1| Amit| 20|   Pune| 45000|
|  2|Pooja| 21| Mumbai| 50000|
|  5|Vikas| 20| Nashik| 53000|
|  4| Neha| 22|Unknown| 48000|
|  6|Sneha| 23|   Pune| 60000|
+---+-----+---+-------+------+



In [ ]:
from pyspark.sql.functions import col, round

# Create a new feature
df = df.withColumn(
    "salary_in_thousands",
    round(col("salary") / 1000, 2)
)

print("Feature processing completed successfully.")
df.show()

Feature processing completed successfully.
+---+-----+---+-------+------+-------------------+
| id| name|age|   city|salary|salary_in_thousands|
+---+-----+---+-------+------+-------------------+
|  3|Rahul| 21|   Pune| 55000|               55.0|
|  1| Amit| 20|   Pune| 45000|               45.0|
|  2|Pooja| 21| Mumbai| 50000|               50.0|
|  5|Vikas| 20| Nashik| 53000|               53.0|
|  4| Neha| 22|Unknown| 48000|               48.0|
|  6|Sneha| 23|   Pune| 60000|               60.0|
+---+-----+---+-------+------+-------------------+



In [ ]:
from pyspark.sql.functions import when

df = df.withColumn(
    "age_group",
    when(col("age") < 21, "Young")
    .when(col("age") <= 22, "Adult")
    .otherwise("Senior")
)

print("Age group feature created successfully.")
df.show()

Age group feature created successfully.
+---+-----+---+-------+------+-------------------+---------+
| id| name|age|   city|salary|salary_in_thousands|age_group|
+---+-----+---+-------+------+-------------------+---------+
|  3|Rahul| 21|   Pune| 55000|               55.0|    Adult|
|  1| Amit| 20|   Pune| 45000|               45.0|    Young|
|  2|Pooja| 21| Mumbai| 50000|               50.0|    Adult|
|  5|Vikas| 20| Nashik| 53000|               53.0|    Young|
|  4| Neha| 22|Unknown| 48000|               48.0|    Adult|
|  6|Sneha| 23|   Pune| 60000|               60.0|   Senior|
+---+-----+---+-------+------+-------------------+---------+



In [ ]:
from pyspark.sql.functions import sum, col

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

print("Final number of records:", df.count())

+---+----+---+----+------+-------------------+---------+
| id|name|age|city|salary|salary_in_thousands|age_group|
+---+----+---+----+------+-------------------+---------+
|  0|   0|  0|   0|     0|                  0|        0|
+---+----+---+----+------+-------------------+---------+

Final number of records: 6


In [ ]:
# Save the final processed DataFrame as CSV
output_path = "/content/final_processed_data"

df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_path)

print("Final processed DataFrame saved successfully as CSV.")

Final processed DataFrame saved successfully as CSV.


In [ ]:
import shutil

shutil.make_archive(
    "/content/final_processed_data",
    "zip",
    "/content/final_processed_data"
)

from google.colab import files
files.download("/content/final_processed_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CONCLUSION

The PySpark DataFrame was successfully created and processed. Missing values were handled using suitable replacement values, and duplicate records were removed to improve data quality. Feature processing was performed by creating salary_in_thousands and age_group features. Final quality checks confirmed that the dataset contained 6 valid records with no missing values. The cleaned and processed DataFrame was successfully saved in CSV format for further analysis and use.